# 03 · Gold — Dimensões | Acidentes ANTT

| | |
|---|---|
| **Origem** | Delta Table `silver_acidentes` |
| **Destino** | Delta Tables `gold_dim_*` |
| **Execução no Pipeline** | Etapa 1 — deve completar antes dos notebooks de fato |
| **Idempotente** | Sim — `overwrite` recria todas as dims |

**Dimensões geradas:**

| Tabela | Linhas esperadas | Chave natural |
|---|---|---|
| `gold_dim_data` | 1 por data de acidente | `data` |
| `gold_dim_concessionaria` | 35 | `concessionaria` |
| `gold_dim_rodovia` | 1 por rodovia+UF | `rodovia, uf` |
| `gold_dim_tipo_acidente` | 1 por tipo | `tipo_de_acidente` |
| `gold_dim_veiculo` | 10 (estática) | `tipo_veiculo` |
| `gold_dim_tipo_vitima` | 5 (estática) | `tipo_vitima` |

## 1 · Imports

In [ ]:
from typing import List

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window

## 2 · Parâmetros

> Célula marcada como **Parameters** — valores podem ser sobrescritos por um Data Pipeline.

In [ ]:
NOTEBOOK_NAME: str = "gold_dims"
TABELA_SILVER: str = "silver_acidentes"
PREFIXO_GOLD:  str = "gold_"
MODO_ESCRITA:  str = "overwrite"
LOG_LEVEL:     str = "INFO"

## 3 · Configuração Compartilhada

In [ ]:
%run 00_nb_config

In [ ]:
# ── Otimizações Spark ─────────────────────────────────────────────────────────
# AQE: replaneja joins e reparticiona resultados em tempo de execucao (Spark 3.3+)
spark.conf.set("spark.sql.adaptive.enabled",                    "true")
# coalescePartitions: AQE consolida particoes pequenas dinamicamente apos shuffles
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Dimensoes <= 100 MB viram broadcast join automaticamente (evita sort-merge join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",          str(100 * 1024 * 1024))
log.info("Spark: AQE=on | coalescePartitions=on | broadcast_threshold=100MB")

## 4 · Leitura Silver

In [ ]:
df = spark.table(TABELA_SILVER)
log.info("Silver lida: %d registros", df.count())

## 5 · Construção das Dimensões

In [ ]:
# Tipos de veiculo e vitima — base para as dims estaticas e para os notebooks de fato
COLS_VEICULOS: List[str] = [
    "automovel", "bicicleta", "caminhao", "moto", "onibus",
    "outros", "tracao_animal", "transporte_de_cargas_especiais",
    "trator_maquinas", "utilitarios",
]
COLS_VITIMAS: List[str] = [
    "ilesos", "levemente_feridos", "moderadamente_feridos",
    "gravemente_feridos", "mortos",
]

# ── dim_data ──────────────────────────────────────────────────────────────────
# SK yyyyMMdd: inteiro legivel, join eficiente, sem lookup extra no Power BI
dim_data = (
    df.select("data")
    .distinct()
    .filter(F.col("data").isNotNull())
    .withColumn("id_data",       F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano",           F.year("data"))
    .withColumn("mes",           F.month("data"))
    .withColumn("dia",           F.dayofmonth("data"))
    .withColumn("trimestre",     F.quarter("data"))
    .withColumn("dia_semana",    F.dayofweek("data"))     # 1=Dom … 7=Sab
    .withColumn("nome_mes",      F.date_format("data", "MMMM"))
    .withColumn("fim_de_semana", F.dayofweek("data").isin(1, 7).cast("boolean"))
    .select("id_data", "data", "ano", "mes", "dia",
            "trimestre", "dia_semana", "nome_mes", "fim_de_semana")
)

# ── dim_concessionaria ────────────────────────────────────────────────────────
dim_concessionaria = (
    df.select("concessionaria")
    .distinct()
    .filter(F.col("concessionaria").isNotNull())
    .withColumn("id_concessionaria",
                F.row_number().over(Window.orderBy("concessionaria")))
    .select("id_concessionaria", "concessionaria")
)

# ── dim_rodovia ───────────────────────────────────────────────────────────────
dim_rodovia = (
    df.select("rodovia", "uf")
    .distinct()
    .filter(F.col("rodovia").isNotNull() & F.col("uf").isNotNull())
    .withColumn("id_rodovia",
                F.row_number().over(Window.orderBy("uf", "rodovia")))
    .select("id_rodovia", "rodovia", "uf")
)

# ── dim_tipo_acidente ─────────────────────────────────────────────────────────
dim_tipo_acidente = (
    df.select("tipo_de_acidente")
    .distinct()
    .filter(F.col("tipo_de_acidente").isNotNull())
    .withColumn("id_tipo_acidente",
                F.row_number().over(Window.orderBy("tipo_de_acidente")))
    .select("id_tipo_acidente", "tipo_de_acidente")
)

# ── dim_veiculo ───────────────────────────────────────────────────────────────
# Dimensao estatica: lista conhecida de tipos de veiculo (nao deriva do silver)
dim_veiculo = spark.createDataFrame(
    [(i + 1, v) for i, v in enumerate(sorted(COLS_VEICULOS))],
    schema="id_veiculo INT, tipo_veiculo STRING",
)

# ── dim_tipo_vitima ───────────────────────────────────────────────────────────
# Dimensao estatica: lista conhecida de tipos de vitima
dim_tipo_vitima = spark.createDataFrame(
    [(i + 1, v) for i, v in enumerate(sorted(COLS_VITIMAS))],
    schema="id_tipo_vitima INT, tipo_vitima STRING",
)

log.info("6 dimensões construídas em memória.")

## 6 · Persistência

In [ ]:
def salvar_gold(df_gold: DataFrame, sufixo: str) -> int:
    """Persiste Delta Table na camada Gold e retorna o total de registros."""
    nome = f"{PREFIXO_GOLD}{sufixo}"
    (
        df_gold.write
        .format("delta")
        .mode(MODO_ESCRITA)
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )
    total = spark.sql(f"SELECT COUNT(*) FROM {nome}").collect()[0][0]
    log.info("%-40s %d registros", nome, total)
    return total


salvar_gold(dim_data,          "dim_data")
salvar_gold(dim_concessionaria,"dim_concessionaria")
salvar_gold(dim_rodovia,       "dim_rodovia")
salvar_gold(dim_tipo_acidente, "dim_tipo_acidente")
salvar_gold(dim_veiculo,       "dim_veiculo")
salvar_gold(dim_tipo_vitima,   "dim_tipo_vitima")

## 7 · Relatório

In [ ]:
DIMS = [
    "dim_data", "dim_concessionaria", "dim_rodovia",
    "dim_tipo_acidente", "dim_veiculo", "dim_tipo_vitima",
]

log.info("=== Dimensões Gold — Resumo ===")
for d in DIMS:
    nome = f"{PREFIXO_GOLD}{d}"
    total = spark.sql(f"SELECT COUNT(*) FROM {nome}").collect()[0][0]
    log.info("  %-40s %d registros", nome, total)

notebookutils.notebook.exit("OK: 6 dimensões Gold criadas com sucesso.")